In [1]:
from sqlalchemy import create_engine
import pandas as pd
from dotenv import load_dotenv
import os

# 환경 변수 로드
load_dotenv()
database_url = os.getenv('DATABASE_URL')
engine = create_engine(database_url)
print("db 연결 성공!")

db 연결 성공!


# interventions table

In [2]:
# 데이터 로드 및 초기 정제

query = 'select nct_id, intervention_type from ctgov.interventions;'
df_internentions = pd.read_sql(query, engine)

In [3]:
df_internentions.info()

<class 'pandas.DataFrame'>
RangeIndex: 972325 entries, 0 to 972324
Data columns (total 2 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   nct_id             972325 non-null  str  
 1   intervention_type  972325 non-null  str  
dtypes: str(2)
memory usage: 14.8 MB


In [4]:
df_internentions.value_counts()

nct_id       intervention_type
NCT02374567  DRUG                 97
NCT02123615  DRUG                 60
NCT00042289  DRUG                 57
NCT03752229  DEVICE               48
NCT04999761  DRUG                 47
                                  ..
NCT07223125  DRUG                  1
NCT07225283  BEHAVIORAL            1
NCT07226531  DRUG                  1
NCT07240766  DRUG                  1
NCT06453928  DEVICE                1
Name: count, Length: 592628, dtype: int64

In [6]:
df_internentions['intervention_type'].unique()

<StringArray>
[              'OTHER',                'DRUG',          'BEHAVIORAL',
             'GENETIC',              'DEVICE',     'DIAGNOSTIC_TEST',
           'PROCEDURE',  'DIETARY_SUPPLEMENT',          'BIOLOGICAL',
 'COMBINATION_PRODUCT',           'RADIATION']
Length: 11, dtype: str

In [7]:
df_internentions['intervention_type'].value_counts()

intervention_type
DRUG                   400154
OTHER                  171101
DEVICE                  98796
BEHAVIORAL              92141
PROCEDURE               87868
BIOLOGICAL              45981
DIETARY_SUPPLEMENT      29415
DIAGNOSTIC_TEST         25917
RADIATION               12537
COMBINATION_PRODUCT      4429
GENETIC                  3986
Name: count, dtype: int64

In [9]:
df_internentions['intervention_type'].isnull().sum()

np.int64(0)

In [11]:
print(df_internentions.shape)

(972325, 2)


In [13]:
# 널값 & 중복값 삭제
df_internentions = df_internentions.dropna(subset=['nct_id', 'intervention_type'])
df_internentions = df_internentions.drop_duplicates()
print(df_internentions.shape)

(592628, 2)


In [14]:
# 소문자로 통일
df_internentions['intervention_type'] = df_internentions['intervention_type'].str.lower()
# 범주형 데이터를 수치형 데이터로 변환 (원핫인코딩)
df_internentions = pd.get_dummies(df_internentions, columns=['intervention_type'], prefix='intervention', dtype=int)
# 여러 치료법을 하나의 행으로 그룹핑
df_internentions = df_internentions.groupby('nct_id').max().reset_index()
df_internentions.head()

,nct_id,intervention_behavioral,intervention_biological,intervention_combination_product,intervention_device,intervention_diagnostic_test,intervention_dietary_supplement,intervention_drug,intervention_genetic,intervention_other,intervention_procedure,intervention_radiation
0,NCT00000102,0,0,0,0,0,0,1,0,0,0,0
1,NCT00000104,0,0,0,0,0,0,0,0,0,1,0
2,NCT00000105,0,1,0,0,0,0,1,0,0,0,0
3,NCT00000106,0,0,0,1,0,0,0,0,0,0,0
4,NCT00000108,1,0,0,0,0,0,0,0,0,0,0


In [15]:
print(df_internentions.shape)

(517127, 12)


In [16]:
# 개별 치료 항목들을 합쳐서 수치형 파생변수 생성
intervention_cols = [col for col in df_internentions.columns if col.startswith('intervention_')]
df_internentions['intervention_count'] = df_internentions[intervention_cols].sum(axis=1)
df_internentions.head()

,nct_id,intervention_behavioral,intervention_biological,intervention_combination_product,intervention_device,intervention_diagnostic_test,intervention_dietary_supplement,intervention_drug,intervention_genetic,intervention_other,intervention_procedure,intervention_radiation,intervention_count
0,NCT00000102,0,0,0,0,0,0,1,0,0,0,0,1
1,NCT00000104,0,0,0,0,0,0,0,0,0,1,0,1
2,NCT00000105,0,1,0,0,0,0,1,0,0,0,0,2
3,NCT00000106,0,0,0,1,0,0,0,0,0,0,0,1
4,NCT00000108,1,0,0,0,0,0,0,0,0,0,0,1


In [17]:
# 복합치료 여부로 이진형 파생변수 생성
df_internentions['has_multiple_intervention_types'] = (df_internentions['intervention_count'] > 1).astype(int)
df_internentions['has_multiple_intervention_types'].value_counts()


has_multiple_intervention_types
0    451934
1     65193
Name: count, dtype: int64

In [ ]:
# 중복값 체크 - 데이터 무결성 검증단계
# assert : 이 조건은 무조건 참이어야 함. 디버깅용 도구로 사용
assert df_internentions['nct_id'].is_unique, 'NCT ID duplication error!'

In [19]:
df_internentions.info()

<class 'pandas.DataFrame'>
RangeIndex: 517127 entries, 0 to 517126
Data columns (total 14 columns):
 #   Column                            Non-Null Count   Dtype
---  ------                            --------------   -----
 0   nct_id                            517127 non-null  str  
 1   intervention_behavioral           517127 non-null  int64
 2   intervention_biological           517127 non-null  int64
 3   intervention_combination_product  517127 non-null  int64
 4   intervention_device               517127 non-null  int64
 5   intervention_diagnostic_test      517127 non-null  int64
 6   intervention_dietary_supplement   517127 non-null  int64
 7   intervention_drug                 517127 non-null  int64
 8   intervention_genetic              517127 non-null  int64
 9   intervention_other                517127 non-null  int64
 10  intervention_procedure            517127 non-null  int64
 11  intervention_radiation            517127 non-null  int64
 12  intervention_count         

In [20]:
# 최종 데이터셋 저장
target_path = r'C:\dev\clinicaltrials-study\data\processed'
file_name = 'interventions_clean.csv'
full_save_path = os.path.join(target_path, file_name)
df_internentions.to_csv(full_save_path, index=False)

In [21]:
df_internentions.head()

,nct_id,intervention_behavioral,intervention_biological,intervention_combination_product,intervention_device,intervention_diagnostic_test,intervention_dietary_supplement,intervention_drug,intervention_genetic,intervention_other,intervention_procedure,intervention_radiation,intervention_count,has_multiple_intervention_types
0,NCT00000102,0,0,0,0,0,0,1,0,0,0,0,1,0
1,NCT00000104,0,0,0,0,0,0,0,0,0,1,0,1,0
2,NCT00000105,0,1,0,0,0,0,1,0,0,0,0,2,1
3,NCT00000106,0,0,0,1,0,0,0,0,0,0,0,1,0
4,NCT00000108,1,0,0,0,0,0,0,0,0,0,0,1,0
